In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

df= spark.read.format("csv").option("header","true").load("/Volumes/agriculture_dev/source/agriculture_dataset/agriculture_dataset.csv")

df.display()

Farm_ID,Crop_Type,Farm_Area(acres),Irrigation_Type,Fertilizer_Used(tons),Pesticide_Used(kg),Yield(tons),Soil_Type,Season,Water_Usage(cubic meters)
F001,Cotton,329.4,Sprinkler,8.14,2.21,14.44,Loamy,Kharif,76648.2
F002,Carrot,18.67,Manual,4.77,4.36,42.91,Peaty,Kharif,68725.54
F003,Sugarcane,306.03,Flood,2.91,0.56,33.44,Silty,Kharif,75538.56
F004,Tomato,380.21,Rain-fed,3.32,4.35,34.08,Silty,Zaid,45401.23
F005,Tomato,135.56,Sprinkler,8.33,4.48,43.28,Clay,Zaid,93718.69
F006,Sugarcane,12.5,Sprinkler,6.42,2.25,38.18,Loamy,Zaid,46487.98
F007,Soybean,360.06,Drip,1.83,2.37,44.93,Sandy,Rabi,40583.57
F008,Rice,464.6,Drip,5.18,0.91,4.23,Silty,Kharif,9392.38
F009,Maize,389.37,Drip,0.57,4.93,3.86,Peaty,Rabi,60202.14
F010,Soybean,184.37,Drip,2.18,2.67,17.25,Sandy,Kharif,90922.15


In [0]:
# ---- Per-column data profile ----
# For every column: row count, distinct count, null/blank count, duplicate count,
# min, max, and invalid values.
# The CSV was read without a schema, so ALL columns are strings:
#   - null/blank: NULL or whitespace-only string
#   - invalid: non-blank value that fails the numeric format check (numeric columns only)
#   - min/max: numeric comparison (cast to double) for numeric columns, string comparison otherwise

numeric_cols = ["Farm_Area(acres)", "Fertilizer_Used(tons)", "Yield(tons)", "Pesticide_Used(kg)", "Water_Usage(cubic meters)"]
valid_number_regex = "^-?[0-9]+(\\.[0-9]+)?$"   # optional sign, digits, optional decimals

row_count = df.count()
profile_rows = []

for c in df.columns:
    v = trim(col(c))
    is_missing = col(c).isNull() | (v == "")

    # numeric columns: non-blank values that are not valid numbers are "invalid"
    # categorical columns: extend here with an expected-domain check if needed
    if c in numeric_cols:
        is_invalid = (~is_missing) & (~v.rlike(valid_number_regex))
    else:
        is_invalid = lit(False)

    # one aggregation pass per column
    stats = df.agg(
        sum(when(is_missing, 1).otherwise(0)).alias("null_count"),
        countDistinct(when(~is_missing, v)).alias("distinct_count"),
        sum(when(is_invalid, 1).otherwise(0)).alias("invalid_count"),
        min(v.cast("double") if c in numeric_cols else v).alias("min_value"),
        max(v.cast("double") if c in numeric_cols else v).alias("max_value"),
    ).first()

    # duplicate occurrences = non-blank rows that repeat an already-counted distinct value
    duplicate_count = (row_count - (stats["null_count"] or 0)) - (stats["distinct_count"] or 0)

    # sample of up to 10 distinct invalid values (numeric columns only)
    invalid_values = (
        df.filter(is_invalid).limit(10).agg(collect_set(c).alias("ivs")).first()["ivs"]
        if c in numeric_cols
        else []
    )

    profile_rows.append((
        c,
        "numeric" if c in numeric_cols else "categorical",
        row_count,
        stats["distinct_count"],
        stats["null_count"],
        duplicate_count,
        None if stats["min_value"] is None else str(stats["min_value"]),
        None if stats["max_value"] is None else str(stats["max_value"]),
        stats["invalid_count"],
        invalid_values,
    ))

profile_df = spark.createDataFrame(profile_rows, schema="""
    column_name string, data_type string, row_count long, distinct_count long, null_count long,
    duplicate_count long, min_value string, max_value string, invalid_count long, invalid_values array<string>
""")

profile_df.display()


profile_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("agriculture_dev.source.source_profile")

column_name,data_type,row_count,distinct_count,null_count,duplicate_count,min_value,max_value,invalid_count,invalid_values
Farm_ID,categorical,55,55,0,0,F001,F055,0,List()
Crop_Type,categorical,55,11,0,44,Barley,Wheat,0,List()
Farm_Area(acres),numeric,55,55,0,0,12.5,483.88,0,List()
Irrigation_Type,categorical,55,6,0,49,Drip,Subsurface,0,List()
Fertilizer_Used(tons),numeric,55,54,0,1,0.5,9.96,0,List()
Pesticide_Used(kg),numeric,55,51,0,4,0.14,4.99,0,List()
Yield(tons),numeric,55,55,0,0,3.86,48.02,0,List()
Soil_Type,categorical,55,5,0,50,Clay,Silty,0,List()
Season,categorical,55,3,0,52,Kharif,Zaid,0,List()
Water_Usage(cubic meters),numeric,55,55,0,0,5869.75,94754.73,0,List()


In [0]:
# Data dictionary: source names are retained in Bronze; canonical names are used in Silver/Gold.
data_dictionary = [
    ("Farm_ID", "Farm natural identifier", "STRING", "Required", "Non-blank", "Trim; uppercase"),
    ("Crop_Type", "Crop grown", "STRING", "Required", "Non-blank", "Trim; title case"),
    ("Farm_Area(acres)", "Farm area in acres", "DOUBLE", "Required", "> 0", "Cast to DOUBLE"),
    ("Irrigation_Type", "Irrigation method", "STRING", "Required", "Non-blank", "Trim; title case"),
    ("Fertilizer_Used(tons)", "Fertilizer used in tons", "DOUBLE", "Required", ">= 0", "Cast to DOUBLE"),
    ("Pesticide_Used(kg)", "Pesticide used in kg", "DOUBLE", "Required", ">= 0", "Cast to DOUBLE"),
    ("Yield(tons)", "Crop yield in tons", "DOUBLE", "Required", ">= 0", "Cast to DOUBLE"),
    ("Soil_Type", "Soil classification", "STRING", "Required", "Non-blank", "Trim; title case"),
    ("Season", "Agricultural season", "STRING", "Required", "Non-blank", "Trim; title case"),
    ("Water_Usage(cubic meters)", "Water consumed in cubic meters", "DOUBLE", "Required", ">= 0", "Cast to DOUBLE"),
]
dd = spark.createDataFrame(data_dictionary, ["source_column", "business_meaning", "target_type", "nullability", "valid_rule", "transformation"])
display(dd)

dd.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "agriculture_dev.source.data_dictionary"
    )

source_column,business_meaning,target_type,nullability,valid_rule,transformation
Farm_ID,Farm natural identifier,STRING,Required,Non-blank,Trim; uppercase
Crop_Type,Crop grown,STRING,Required,Non-blank,Trim; title case
Farm_Area(acres),Farm area in acres,DOUBLE,Required,> 0,Cast to DOUBLE
Irrigation_Type,Irrigation method,STRING,Required,Non-blank,Trim; title case
Fertilizer_Used(tons),Fertilizer used in tons,DOUBLE,Required,>= 0,Cast to DOUBLE
Pesticide_Used(kg),Pesticide used in kg,DOUBLE,Required,>= 0,Cast to DOUBLE
Yield(tons),Crop yield in tons,DOUBLE,Required,>= 0,Cast to DOUBLE
Soil_Type,Soil classification,STRING,Required,Non-blank,Trim; title case
Season,Agricultural season,STRING,Required,Non-blank,Trim; title case
Water_Usage(cubic meters),Water consumed in cubic meters,DOUBLE,Required,>= 0,Cast to DOUBLE
